In [1]:
!pip install gymnasium numpy matplotlib

import numpy as np
print("Setup complete!")
print(f"NumPy version: {np.__version__}")

Setup complete!
NumPy version: 2.0.2


In [2]:
import numpy as np

# Simple grid environment
class SimpleChipEnv:
    def __init__(self, grid_size=10):
        self.grid_size = grid_size
        self.grid = np.zeros((grid_size, grid_size))
        self.macros_placed = []

    def reset(self):
        self.grid = np.zeros((self.grid_size, self.grid_size))
        self.macros_placed = []
        return self.grid

    def step(self, action):
        # action = (x, y) position to place macro
        x, y = action
        if self.grid[x, y] == 0:
            self.grid[x, y] = 1
            self.macros_placed.append((x, y))
            # Reward = negative wirelength (simplified)
            reward = -len(self.macros_placed)
            done = len(self.macros_placed) >= 5  # stop after 5 macros
            return self.grid, reward, done
        else:
            return self.grid, -10, False  # penalty for invalid placement

# Random agent
def random_action(env):
    x = np.random.randint(0, env.grid_size)
    y = np.random.randint(0, env.grid_size)
    return (x, y)

# Test the environment
env = SimpleChipEnv(grid_size=10)
state = env.reset()
total_reward = 0

for step in range(10):
    action = random_action(env)
    next_state, reward, done = env.step(action)
    total_reward += reward
    print(f"Step {step+1}: Placed at {action}, Reward: {reward}")
    if done:
        break

print(f"\nTotal Reward: {total_reward}")

Step 1: Placed at (6, 2), Reward: -1
Step 2: Placed at (8, 3), Reward: -2
Step 3: Placed at (3, 9), Reward: -3
Step 4: Placed at (8, 3), Reward: -10
Step 5: Placed at (0, 7), Reward: -4
Step 6: Placed at (4, 2), Reward: -5

Total Reward: -25


In [3]:
import numpy as np

class SimpleChipEnv:
    def __init__(self, grid_size=8, num_macros=5):
        self.grid_size = grid_size
        self.num_macros = num_macros
        self.reset()

    def calculate_wirelength(self, positions):
        if len(positions) < 2:
            return 0
        total = 0
        for i in range(len(positions) - 1):
            x1, y1 = positions[i]
            x2, y2 = positions[i + 1]
            total += abs(x1 - x2) + abs(y1 - y2)
        return total

    def reset(self):
        self.grid = np.zeros((self.grid_size, self.grid_size))
        self.macros_placed = []
        self.step_count = 0
        return self.grid

    def step(self, action):
        x, y = action
        if 0 <= x < self.grid_size and 0 <= y < self.grid_size and self.grid[x, y] == 0:
            self.grid[x, y] = 1
            self.macros_placed.append((x, y))
            self.step_count += 1

            # Calculate wirelength reward
            wirelength = self.calculate_wirelength(self.macros_placed)
            reward = -wirelength  # Shorter wirelength = higher reward

            done = len(self.macros_placed) >= self.num_macros
            return self.grid, reward, done
        else:
            return self.grid, -10, False

    def render(self):
        print("Current Grid:")
        for row in self.grid:
            print(' '.join(['X' if cell == 1 else '.' for cell in row]))
        print(f"Macros placed: {len(self.macros_placed)}/{self.num_macros}")
        print(f"Wirelength: {self.calculate_wirelength(self.macros_placed)}")
        print()

# Random Agent
class RandomAgent:
    def __init__(self, env):
        self.env = env

    def get_action(self):
        x = np.random.randint(0, self.env.grid_size)
        y = np.random.randint(0, self.env.grid_size)
        return (x, y)

# Run the test
print("=" * 50)
print("Testing Random Agent with Wirelength Reward")
print("=" * 50)

env = SimpleChipEnv(grid_size=8, num_macros=5)
agent = RandomAgent(env)

state = env.reset()
env.render()

total_reward = 0

for step in range(5):
    action = agent.get_action()
    next_state, reward, done = env.step(action)
    total_reward += reward
    print(f"Step {step+1}: Placed at {action}, Reward: {reward}")

env.render()
print(f"Total Reward: {total_reward}")

Testing Random Agent with Wirelength Reward
Current Grid:
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Macros placed: 0/5
Wirelength: 0

Step 1: Placed at (2, 2), Reward: 0
Step 2: Placed at (4, 4), Reward: -4
Step 3: Placed at (4, 1), Reward: -7
Step 4: Placed at (0, 4), Reward: -14
Step 5: Placed at (4, 4), Reward: -10
Current Grid:
. . . . X . . .
. . . . . . . .
. . X . . . . .
. . . . . . . .
. X . . X . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Macros placed: 4/5
Wirelength: 14

Total Reward: -35


In [4]:
import numpy as np
import random

class QLearningAgent:
    def __init__(self, env, learning_rate=0.1, discount_factor=0.9, exploration_rate=0.3):
        self.env = env
        self.lr = learning_rate      # How fast to learn
        self.gamma = discount_factor # How much to care about future rewards
        self.epsilon = exploration_rate # How often to try random actions

        # Q-table: stores value of each (state, action)
        # State is simplified as tuple of positions for demo
        self.q_table = {}

    def get_state_key(self):
        """Convert current macro positions to a hashable key"""
        return tuple(sorted(self.env.macros_placed))

    def get_action(self):
        """Choose action using epsilon-greedy strategy"""
        state_key = self.get_state_key()

        # Explore: take random action
        if random.random() < self.epsilon:
            x = random.randint(0, self.env.grid_size - 1)
            y = random.randint(0, self.env.grid_size - 1)
            return (x, y)

        # Exploit: take best known action
        if state_key not in self.q_table:
            self.q_table[state_key] = {}

        # If no actions learned yet, take random
        if len(self.q_table[state_key]) == 0:
            x = random.randint(0, self.env.grid_size - 1)
            y = random.randint(0, self.env.grid_size - 1)
            return (x, y)

        # Find action with highest Q-value
        best_action = max(self.q_table[state_key], key=self.q_table[state_key].get)
        return best_action

    def update(self, state, action, reward, next_state, done):
        """Update Q-table using Bellman equation"""
        state_key = tuple(sorted(state))
        next_state_key = tuple(sorted(next_state))

        # Initialize if not exists
        if state_key not in self.q_table:
            self.q_table[state_key] = {}
        if action not in self.q_table[state_key]:
            self.q_table[state_key][action] = 0

        # Calculate max future Q-value
        max_future_q = 0
        if next_state_key in self.q_table and len(self.q_table[next_state_key]) > 0:
            max_future_q = max(self.q_table[next_state_key].values())

        # Bellman equation: Q(s,a) = Q(s,a) + lr * [reward + gamma * maxQ(s') - Q(s,a)]
        old_q = self.q_table[state_key][action]
        new_q = old_q + self.lr * (reward + self.gamma * max_future_q - old_q)
        self.q_table[state_key][action] = new_q

# Train Q-learning agent
print("=" * 50)
print("Training Q-Learning Agent")
print("=" * 50)

env = SimpleChipEnv(grid_size=6, num_macros=4)  # Smaller for faster training
agent = QLearningAgent(env)

num_episodes = 100
episode_rewards = []

for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.get_action()
        next_state, reward, done = env.step(action)

        # Update Q-table
        agent.update(env.macros_placed[:-1], action, reward, env.macros_placed, done)

        total_reward += reward

    episode_rewards.append(total_reward)

    # Print progress every 20 episodes
    if (episode + 1) % 20 == 0:
        avg_reward = np.mean(episode_rewards[-20:])
        print(f"Episode {episode+1}: Average Reward = {avg_reward:.2f}")

print("\nTraining Complete!")
print(f"Best reward achieved: {max(episode_rewards)}")
print(f"Average reward over last 20 episodes: {np.mean(episode_rewards[-20:]):.2f}")

Training Q-Learning Agent
Episode 20: Average Reward = -27.60
Episode 40: Average Reward = -28.10
Episode 60: Average Reward = -20.55
Episode 80: Average Reward = -30.70
Episode 100: Average Reward = -23.20

Training Complete!
Best reward achieved: -9
Average reward over last 20 episodes: -23.20
